In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

# Initialize Spark with Kafka package
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3"
    ) \
    .getOrCreate()

# 1. Load Static User Data (CSV)
users_df = spark.read.csv(
    "data/user_table.csv",
    header=True,
    inferSchema=True
)

# 2. Read Streaming Data from Kafka
tx_schema = StructType([
    StructField("tx_id", IntegerType(), True),
    StructField("userId", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", DoubleType(), True)
])

kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "fraud-detection") \
    .load()

# 3. Parse and Filter Data
parsed_stream = kafka_stream.select(
    from_json(
        col("value").cast("string"),
        tx_schema
    ).alias("tx")
).select("tx.*")

fraud_stream = parsed_stream.filter(
    col("amount") > 5000.0
)

# 4. Enrich Stream with User Details
enriched_fraud = fraud_stream.join(
    users_df,
    "userId"
)

# 5. Format for Output Kafka Topic
output_stream = enriched_fraud \
    .withColumn(
        "value",
        to_json(struct("*")).cast("string")
    ) \
    .select("value")

# 6. Write Stream to 'fraud-notification' Topic
query = output_stream.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "fraud-notification") \
    .option(
        "checkpointLocation",
        "/workspace/checkpoints"
    ) \
    .start()

query.awaitTermination()

26/06/19 06:18:52 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 06:18:56 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/06/19 06:18:57 WARN CheckpointFileManager: Failed to rename temp file file:/workspace/checkpoints/offsets/.506.82cf7866-ee6a-4b27-a4fa-af356cf6d917.tmp to file:/workspace/checkpoints/offsets/506 because file exists
org.apache.hadoop.fs.FileAlreadyExistsException: rename destination file:/workspace/checkpoints/offsets/506 already exists.
	at org.apache.hadoop.fs.FileSystem.rename(FileSystem.java:1600)
	at org.apache.hadoop.fs.DelegateToFileSystem.renameInternal(DelegateToFileSystem.java:206)
	at org.apache.hadoop.fs.AbstractFileSystem.renameInternal(AbstractFileSystem.java:790)
	at org.apache.hadoop.fs.AbstractFileSystem.rename(AbstractFileSystem.ja

StreamingQueryException: [STREAM_FAILED] Query [id = 6f1b00e6-411c-4e15-af4e-9419011ea850, runId = 7aa52467-3c64-4c3e-8eb7-9e43dde79c58] terminated with exception: Multiple streaming queries are concurrently using file:/workspace/checkpoints/offsets.